In [52]:
import os
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set the OPENAI_API_KEY environment variable before running this notebook.")

In [53]:
import os
!pip install -q -U youtube-transcript-api langchain-community langchain-openai langgraph faiss-cpu tiktoken python-dotenv requests==2.32.4

In [54]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

**Step 1a - Indexing (Document Ingestion)**

In [103]:
video_id = "Gfr50f6ZBvo"
try:
    # Instantiate the API client
    api_client = YouTubeTranscriptApi()

    # List all available transcripts for this video
    transcript_list = api_client.list(video_id)

    try:
        # Try retrieving the English transcript directly first
        transcript_data = transcript_list.find_transcript(['en']).fetch()
    except Exception:
        # Fallback: Find and retrieve the available native auto-generated transcript (e.g. Hindi)
        transcript_data = transcript_list.find_transcript(['hi']).fetch()

    # Flatten the fetched transcript entries to plain text using attribute access
    transcript = " ".join(chunk.text for chunk in transcript_data)
    print("Successfully fetched transcript! Preview:")
    print(transcript[:1000] + "...")

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred during execution: {e}")

Successfully fetched transcript! Preview:
the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to

In [104]:
transcript_list

**Step 1b - Indexing (Text Splitting)**

In [105]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [106]:
len(chunks)

168

In [107]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

**Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)**

In [108]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

In [109]:
vector_store.index_to_docstore_id

{0: 'b4e96529-8eb8-4ac3-a822-e42be9d738d9',
 1: '5b570a3a-0770-4e99-9dab-8c40caad165c',
 2: '9c046d2e-623d-4f88-b69c-c2fde9554849',
 3: '206431df-4880-45a6-b034-e82047b3cc96',
 4: '57505d58-4897-46c1-9e3c-cc0970b1fa8e',
 5: '96bae469-bb68-438e-b5f5-832d82b7542e',
 6: '690b3d3d-e9a2-4c28-afeb-9813b10a1b08',
 7: '4fa387b1-14ec-48d4-8746-4f11a2f69dfb',
 8: 'aef13f4b-24e8-4f3c-81c0-fed3e8c9b0aa',
 9: '8bc79603-167f-41a7-8754-1b4b131bd4b5',
 10: '283c4e93-93a3-47a2-8b8c-5c5240f6ea5c',
 11: 'd7cdb751-45c2-46ca-a1e0-80bdbe07eb08',
 12: '5e2b9b08-415c-4fb7-af7c-6c8639f46ce3',
 13: 'aa32449a-79d6-487c-bc87-a9652c52f4fb',
 14: '35b5dc23-1a2e-4f01-ba23-7851bf354409',
 15: '795a2b2b-35da-4f94-bd02-8ed5efb22edb',
 16: 'd33570c0-db05-4222-af53-2bcdc44edb67',
 17: '08b9a18c-8095-4424-afc9-039eca60791a',
 18: '4e0792e8-9359-430f-84be-e1e73e8cc758',
 19: '9cc36d06-9a53-4b25-afb5-adc0f044829d',
 20: '7f827464-e5fa-41c9-84ab-ebe8e5eca9b2',
 21: 'f129e0ca-ee27-4d74-a66a-94963a296bab',
 22: 'fc925818-ebd4-

In [110]:
vector_store.get_by_ids(['7e873b73-d93f-439a-bb95-ce93e5dcc7e7'])

[]

**Step 2 - Retrieval**

In [111]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [112]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7dd49e5434d0>, search_kwargs={'k': 4})

In [113]:
retriever.invoke('What is deepmind')

[Document(id='b4e96529-8eb8-4ac3-a822-e42be9d738d9', metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal qu

***Step 3 - Augmentation**

In [114]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [115]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [118]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [119]:
retrieved_docs

[Document(id='6bc70f5b-324e-4292-8b17-12d9a72fbf91', metadata={}, page_content="so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals by solving the fractional electron problem so you're taking on modeling and simulating the quantum mechanical 

In [120]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals by solving the fractional electron problem so you're taking on modeling and simulating the quantum mechanical behavior of electrons yes um can you explain this work and can ai model and\n\n

In [122]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [123]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals

**Step 4 - Generation**

In [124]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of nuclear fusion is discussed in the video. The discussion includes the following points:

1. The speaker mentions a problem in fusion that was published in a Nature paper, where they developed a controller to hold plasma in specific shapes for a record amount of time, which is significant for energy production.

2. They talk about collaborating with EPFL in Switzerland, which has a test reactor used for various experiments, and how they identify bottleneck problems in fusion that could be addressed using AI methods.

3. The speaker emphasizes the potential of AI to help accelerate solutions in energy and climate, particularly in fusion, which faces challenges in physics, material science, and engineering.

4. They mention a specific paper on nuclear fusion that involves using deep reinforcement learning for the magnetic control of tokamak plasmas, indicating an effort to solve nuclear fusion challenges through AI.


**Building a Chain**

In [125]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [126]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [127]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [128]:
parallel_chain.invoke('Give me the movies details')

{'context': "do you wake up are you night owl do you wake up early in the morning what are some interesting habits uh how many dozens of cups of coffees do you drink a day what's the computer um that you use uh what's the setup how many screens what kind of keyboard are we talking uh emacs vim are we talking something more modern so it's a bunch of those questions so maybe uh day in the life what what's the perfect day involved well these days it's quite different from say 10 20 years ago back 10 20 years ago it would have been you know a whole day of research individual research or programming doing some experiment neuroscience computer science experiment reading lots of research papers uh and then perhaps at night time you know um reading science fiction books or or uh playing uh some games but lots of focus so like deep focused work on whether it's uh programming or reading research paper yes yes so that would be a lot of debrief you know uh focused work these days for the last sort

In [129]:
parser = StrOutputParser()

In [130]:
main_chain = parallel_chain | prompt | llm | parser

In [131]:
main_chain.invoke('Can you summarize the video')

'The video features a conversation with Demas, who discusses the limitations of the standard model of physics and the need for deeper, simpler explanations of fundamental concepts like consciousness, life, and gravity. Demas shares insights on recent advancements in fusion research, including a method for holding plasma in specific shapes to improve energy production. He also touches on the challenges of modeling quantum mechanical behavior of electrons and the potential role of AI in this area. The conversation emphasizes the importance of exploring new ideas and solutions in physics and technology.'